### Import Dependencies

In [1]:
list.of.packages <- c("ggplot2", "Rcpp", "grf", "caret", "mltools", "rpart", "minpack.lm", "doParallel", "rattle", "anytime","rlist")
list.of.packages <- c(list.of.packages, "zoo", "dtw", "foreach", "evaluate","rlist","data.table","plyr","dplyr")


new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='/home/zwang937/local/R_libs', repos="http://cran.us.r-project.org", dependencies = TRUE, INSTALL_opts = '--no-lock')


lapply(list.of.packages, require, character.only = TRUE)

Loading required package: ggplot2

Loading required package: Rcpp

Loading required package: grf

Loading required package: caret

Loading required package: lattice

Loading required package: mltools

Loading required package: rpart

Loading required package: minpack.lm

Loading required package: doParallel

Loading required package: foreach

Loading required package: iterators

Loading required package: parallel

Loading required package: rattle

Loading required package: tibble

Loading required package: bitops

Rattle: A free graphical interface for data science with R.
Version 5.5.1 Copyright (c) 2006-2021 Togaware Pty Ltd.
Type 'rattle()' to shake, rattle, and roll your data.

Loading required package: anytime

Loading required package: rlist

Loading required package: zoo


Attaching package: ‘zoo’


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric


Loading required package: dtw

Loading required package: proxy


Attaching package: ‘proxy’


Th

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] TRUE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

[[9]]
[1] TRUE

[[10]]
[1] TRUE

[[11]]
[1] TRUE

[[12]]
[1] TRUE

[[13]]
[1] TRUE

[[14]]
[1] TRUE

[[15]]
[1] TRUE

[[16]]
[1] TRUE

[[17]]
[1] TRUE

[[18]]
[1] TRUE

[[19]]
[1] TRUE

In [2]:
# SET PARAMS
windowsize=2
num_trees = 100



# CREATE OUTPUT FOLDERS
mainDir = "./time_variant_grf_results"
dir.create(mainDir, showWarnings = FALSE)

subDir = paste("time_variant_grf_backtest_state_forests_windowsize=",toString(windowsize),"_numtrees=",toString(num_trees),sep="")
outputfolder = file.path(mainDir, subDir)


dir.create(outputfolder, showWarnings = FALSE)

grf.subfolder = paste("time_variant_grf_grf_windowsize=",toString(windowsize),"_numtrees=",toString(num_trees),sep="")
grf.outputfolder = file.path(mainDir,grf.subfolder)
dir.create(grf.outputfolder, showWarnings = FALSE)


### Read in Panel Data

In [3]:
# READ DATA
print("Reading in data")
destfile <- paste("../../data/augmented_us-counties-states_latest_variants",".csv",sep="")
augmented_panel_data <- as.data.frame(fread(file = destfile))
# Time variant
#hhs_X_w_clusters_fpath = "../benchmark_tcv_kmeans_code/hhs_X_w_clusters.csv"
#hhs_X_w_clusters = as.data.frame(fread(file = hhs_X_w_clusters_fpath))

print("Transforming augmented_panel_data")
augmented_panel_data <- augmented_panel_data[order(augmented_panel_data$fips, augmented_panel_data$days_from_start), ]
augmented_panel_data <- augmented_panel_data[augmented_panel_data$rolled_cases >= 20,]
augmented_panel_data$log_rolled_cases <- log(augmented_panel_data$rolled_cases + 1.1)
augmented_panel_data <- augmented_panel_data %>%
  group_by(fips) %>%
    mutate(log_rolled_cases_ratios = c(0, diff(log_rolled_cases)))
augmented_panel_data <- augmented_panel_data %>%
  group_by(fips) %>%
    mutate(shifted_days_from_start = days_from_start - first(days_from_start))
augmented_panel_data <- augmented_panel_data[, colSums(!is.na(augmented_panel_data)) > 0]

# Obtain the time invariant data specific to each fips
#X_time_invariant <- (hhs_X_w_clusters[,intersect(names(augmented_panel_data), names(hhs_X_w_clusters))])
#X_time_invariant <- X_time_invariant[,!names(X_time_invariant) %in% c("county","state")]



# SET START AND END
start_day = min(augmented_panel_data$days_from_start)
end_day = max(augmented_panel_data$days_from_start)

cutoff_list <- start_day:end_day

[1] "Reading in data"
[1] "Transforming augmented_panel_data"


In [6]:
cutoff = 564
check.file.name <- paste0("classical_grf_stateforest_cutoff=", toString(cutoff), ".rds")
check.file.full.name <- file.path(grf.outputfolder, check.file.name)
if (file.exists(check.file.full.name)){
    print(paste0(check.file.name, " exists, skipping"))
    #next
}
print(paste0("Computing Classical GRF for ", toString(cutoff)))
try({ # START TRY
    start_time <- Sys.time()
    # Given my current cutoff, which block numbers should I use?
    fname <- paste("time_variant_grf_",toString(cutoff),".csv",sep="")        
    # SLICE THE DATA ACCORDINGLY
    early_data <- augmented_panel_data[augmented_panel_data$days_from_start <= cutoff, ]
    case_number_columns <- c("fips","date","county","state","days_from_start","log_rolled_cases", "shifted_days_from_start")
    early_data_case_numbers <- early_data[, names(early_data) %in% case_number_columns]
    # Format data to be fed to GRF
    #WYX <- merge(early_data_case_numbers, X_time_invariant, by="fips", all.x=TRUE)
    WYX <- early_data
    WYX <- WYX[order(WYX$fips, WYX$days_from_start),]

    Y <- WYX$log_rolled_cases
    W <- WYX$shifted_days_from_start
    X <- WYX[, !names(WYX) %in% case_number_columns]
    X <- X %>%
      select_if(is.numeric)

    cf <- causal_forest(X,Y,as.numeric(W), num.trees=num_trees)
    cf.fname <- check.file.name
    cf.path <- file.path(grf.outputfolder, cf.fname)
    print(paste0("Saving ", cf.fname))
    #saveRDS(cf, cf.path)

    # GENERATE r_GRF estimates
    WYX_test <- WYX %>%
      group_by(fips) %>%
      filter(days_from_start == cutoff)
    X_test <- WYX_test[, !names(WYX_test) %in% case_number_columns]
    X_test <- X_test %>%
      select_if(is.numeric)
    r_GRF <- predict(cf, X_test)

    # Generate predictions
    indexing_columns <- c("fips","county","state","date", "days_from_start","shifted_days_from_start", "rolled_cases", "log_rolled_cases")
    indexing <- WYX_test[, names(WYX) %in% indexing_columns]
    indexing$r_GRF <- unlist(r_GRF)
    indexing$GRF_predicted_log_rolled_cases <- indexing$r_GRF*7 + indexing$log_rolled_cases

    output.fname = paste("time_variant_grf_block_results_",toString(cutoff),".csv",sep="")
    destpath = file.path(outputfolder,output.fname)
    #fwrite(indexing, destpath, row.names=FALSE)
    end_time <- Sys.time()
} # END TRY
) # END TRY

[1] "Computing Classical GRF for 564"
[1] "Saving classical_grf_stateforest_cutoff=564.rds"


In [7]:
indexing

fips,date,county,state,days_from_start,rolled_cases,log_rolled_cases,shifted_days_from_start,r_GRF,GRF_predicted_log_rolled_cases
<dbl>,<IDate>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1001,2021-08-07,Autauga,Alabama,564,329.57143,5.801125,478,0.0016938048,5.812982
1003,2021-08-07,Baldwin,Alabama,564,3789.00000,8.240148,490,0.0001241522,8.241017
1005,2021-08-07,Barbour,Alabama,564,180.85714,5.203771,472,0.0008860441,5.209973
1007,2021-08-07,Bibb,Alabama,564,235.57143,5.466673,476,0.0008393291,5.472548
1009,2021-08-07,Blount,Alabama,564,421.00000,6.045242,470,0.0011102096,6.053014
1013,2021-08-07,Butler,Alabama,564,156.71429,5.061419,467,0.0001363751,5.062374
1015,2021-08-07,Calhoun,Alabama,564,708.28571,6.564399,487,0.0008190283,6.570133
1017,2021-08-07,Chambers,Alabama,564,244.85714,5.505157,494,0.0013789374,5.514810
1019,2021-08-07,Cherokee,Alabama,564,121.14286,4.806010,414,0.0009236261,4.812475


In [8]:
X_test

cases,deaths,rolled_cases,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,⋯,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.16,Variant % XBB.1.5,Variant % XBB.1.5.1,Variant % XBB.1.9.1,Variant % XBB.1.9.2,Variant % XBB.2.3,log_rolled_cases_ratios
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
404,114,329.57143,32.53224,-86.64644,594.4435,55200,23315,21115,8422,⋯,0,0,0,0,0,0,0,0,0,0.07209280
4456,330,3789.00000,30.65922,-87.74607,1589.7930,208107,111945,78622,21653,⋯,0,0,0,0,0,0,0,0,0,0.05696177
224,63,180.85714,31.87025,-85.40510,885.0016,25782,11937,9186,6597,⋯,0,0,0,0,0,0,0,0,0,0.06403094
271,66,235.57143,33.01589,-87.12715,622.4611,22527,9161,6840,2863,⋯,0,0,0,0,0,0,0,0,0,0.04506379
494,140,421.00000,33.97736,-86.56644,644.8305,57645,24222,20600,8220,⋯,0,0,0,0,0,0,0,0,0,0.05567236
189,72,156.71429,31.75167,-86.68197,776.8382,20025,10026,6708,4640,⋯,0,0,0,0,0,0,0,0,0,0.05680171
862,336,708.28571,33.77052,-85.82791,605.8673,115098,53682,45033,20819,⋯,0,0,0,0,0,0,0,0,0,0.06188783
307,125,244.85714,32.91550,-85.39403,596.5606,33826,16981,13516,5531,⋯,0,0,0,0,0,0,0,0,0,0.07725382
149,48,121.14286,34.06952,-85.65424,553.5248,25853,16531,10606,3827,⋯,0,0,0,0,0,0,0,0,0,0.06893444


In [9]:
X <- WYX[, !names(WYX) %in% case_number_columns]

In [10]:
X <- X[, !names(X) %in% c("cases","deaths","datetime","rolled_cases")]

In [11]:
dim(X)

[1] 1095276     470

In [12]:
X_numeric <- X %>%
  select_if(is.numeric)

In [13]:
as.numeric(X_numeric)

ERROR: Error in eval(expr, envir, enclos): 'list' object cannot be coerced to type 'double'


In [19]:
for (i in seq_along(colnames(X))) {
  cat("Column ", i, " (", colnames(X)[i], ")\n")
}

Column  1  ( LAT )
Column  2  ( LON )
Column  3  ( AREA_SQMI )
Column  4  ( E_TOTPOP )
Column  5  ( E_HU )
Column  6  ( E_HH )
Column  7  ( E_POV )
Column  8  ( E_UNEMP )
Column  9  ( E_PCI )
Column  10  ( E_NOHSDP )
Column  11  ( E_AGE65 )
Column  12  ( E_AGE17 )
Column  13  ( E_DISABL )
Column  14  ( E_SNGPNT )
Column  15  ( E_MINRTY )
Column  16  ( E_LIMENG )
Column  17  ( E_MUNIT )
Column  18  ( E_MOBILE )
Column  19  ( E_CROWD )
Column  20  ( E_NOVEH )
Column  21  ( E_GROUPQ )
Column  22  ( EP_POV )
Column  23  ( MP_POV )
Column  24  ( EP_UNEMP )
Column  25  ( MP_UNEMP )
Column  26  ( EP_PCI )
Column  27  ( MP_PCI )
Column  28  ( EP_NOHSDP )
Column  29  ( MP_NOHSDP )
Column  30  ( EP_AGE65 )
Column  31  ( MP_AGE65 )
Column  32  ( EP_AGE17 )
Column  33  ( MP_AGE17 )
Column  34  ( EP_DISABL )
Column  35  ( MP_DISABL )
Column  36  ( EP_SNGPNT )
Column  37  ( MP_SNGPNT )
Column  38  ( EP_MINRTY )
Column  39  ( MP_MINRTY )
Column  40  ( EP_LIMENG )
Column  41  ( MP_LIMENG )
Column  42 

In [20]:
# 102 State_of_emergency_issued
# 358 Mention_of_tribal_casinos

In [24]:
X_removed <- X[, -(102:358)]
for (i in seq_along(colnames(X_removed))) {
  cat("Column ", i, " (", colnames(X_removed)[i], ")\n")
}

Column  1  ( LAT )
Column  2  ( LON )
Column  3  ( AREA_SQMI )
Column  4  ( E_TOTPOP )
Column  5  ( E_HU )
Column  6  ( E_HH )
Column  7  ( E_POV )
Column  8  ( E_UNEMP )
Column  9  ( E_PCI )
Column  10  ( E_NOHSDP )
Column  11  ( E_AGE65 )
Column  12  ( E_AGE17 )
Column  13  ( E_DISABL )
Column  14  ( E_SNGPNT )
Column  15  ( E_MINRTY )
Column  16  ( E_LIMENG )
Column  17  ( E_MUNIT )
Column  18  ( E_MOBILE )
Column  19  ( E_CROWD )
Column  20  ( E_NOVEH )
Column  21  ( E_GROUPQ )
Column  22  ( EP_POV )
Column  23  ( MP_POV )
Column  24  ( EP_UNEMP )
Column  25  ( MP_UNEMP )
Column  26  ( EP_PCI )
Column  27  ( MP_PCI )
Column  28  ( EP_NOHSDP )
Column  29  ( MP_NOHSDP )
Column  30  ( EP_AGE65 )
Column  31  ( MP_AGE65 )
Column  32  ( EP_AGE17 )
Column  33  ( MP_AGE17 )
Column  34  ( EP_DISABL )
Column  35  ( MP_DISABL )
Column  36  ( EP_SNGPNT )
Column  37  ( MP_SNGPNT )
Column  38  ( EP_MINRTY )
Column  39  ( MP_MINRTY )
Column  40  ( EP_LIMENG )
Column  41  ( MP_LIMENG )
Column  42 